# Customer Churn Prediction — Pipeline Notebook
Mirrors `main.py` step-by-step. Each cell maps to one `src/` module.

## 0. Setup

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

# Point to src/ so notebook imports work identically to main.py
sys.path.insert(0, os.path.join('..', 'src'))

RAW_PATH = os.path.join('..', 'data', 'raw', 'telco_churn.csv')

## 1. Load & Clean  (`src/preprocess.py → load_and_clean`)

In [ ]:
from preprocess import load_and_clean

df_raw = load_and_clean(RAW_PATH)

print(f'Shape      : {df_raw.shape}')
print(f'Churn rate : {df_raw["Churn"].mean()*100:.1f}%')
df_raw.head(3)

## 2. Feature Engineering  (`src/preprocess.py → engineer_features`)

In [ ]:
from preprocess import engineer_features

df_feat = engineer_features(df_raw)

# Confirm new columns exist
new_cols = ['avg_monthly_spend', 'num_services', 'high_value', 'tenure_group']
print('Engineered features added:')
df_feat[new_cols].describe()

## 3. Encode & Split  (`src/preprocess.py → encode_and_split`)

In [ ]:
from preprocess import encode_and_split

(
    X_train, X_test, y_train, y_test,
    X_train_sc, X_test_sc,
    scaler, imputer,
    feature_names,
) = encode_and_split(df_feat)

print(f'X_train : {X_train.shape}   X_test : {X_test.shape}')
print(f'y_train churn rate : {y_train.mean()*100:.1f}%')
print(f'y_test  churn rate : {y_test.mean()*100:.1f}%')

## 4. Train Models  (`src/train.py → train_all`)

SMOTE is applied **inside** `train_all` on `X_train_sc` only — test set is never touched.

In [ ]:
from train import train_all
import pandas as pd

results, best_name = train_all(X_train_sc, y_train, X_test_sc, y_test)

# Summary table
rows = []
for name, r in results.items():
    rows.append({
        'Model'        : name,
        'ROC-AUC'      : round(r['roc_auc'], 4),
        'Avg Precision': round(r['avg_precision'], 4),
        'F1 (Churn)'   : round(r['report']['1']['f1-score'], 4),
    })

comp = pd.DataFrame(rows).set_index('Model')
comp.style.highlight_max(color='lightgreen', axis=0)

In [ ]:
print(f'Best model: {best_name}  (ROC-AUC {results[best_name]["roc_auc"]:.4f})')
best = results[best_name]

## 5. Evaluate Best Model

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay(best['cm'], display_labels=['Retained', 'Churned']).plot(
    ax=axes[0], colorbar=False, cmap='Blues'
)
axes[0].set_title('Confusion Matrix')

RocCurveDisplay.from_predictions(
    y_test, best['proba'], name=best_name, ax=axes[1]
)
axes[1].set_title('ROC Curve')

plt.tight_layout()
plt.show()

In [ ]:
# Precision-Recall curve (more informative under class imbalance)
from sklearn.metrics import PrecisionRecallDisplay

PrecisionRecallDisplay.from_predictions(y_test, best['proba'], name=best_name)
plt.title('Precision-Recall Curve')
plt.show()

## 6. Feature Importance  (`src/train.py → get_feature_importance`)

In [ ]:
from train import get_feature_importance

fi_df = get_feature_importance(best['model'], feature_names)

top12 = fi_df.head(12)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top12['feature'][::-1], top12['importance'][::-1])
ax.set_xlabel('Importance')
ax.set_title('Top 12 Churn Drivers')
plt.tight_layout()
plt.show()

fi_df.head(12)

## 7. Score All Customers  (`src/score.py → score_all_customers`)

In [ ]:
from score import score_all_customers
from sklearn.preprocessing import LabelEncoder

# Re-encode full dataset (same logic as encode_and_split but no split)
df_enc = engineer_features(load_and_clean(RAW_PATH)).drop(columns=['customerID', 'Churn'])
le = LabelEncoder()
for col in df_enc.select_dtypes(include=['object', 'category']).columns:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))

scored = score_all_customers(df_raw, best['model'], scaler, imputer, df_enc)

print(f'Scored {len(scored):,} customers')
scored.head(10)

## 8. Revenue at Risk  (`src/score.py → revenue_summary`)

In [ ]:
from score import revenue_summary

summary = revenue_summary(scored, best['roc_auc'])

print(f"Total monthly revenue   : {summary['total_rev']:>12,.0f}")
print(f"High-risk monthly rev   : {summary['rev_at_risk']:>12,.0f}")
print(f"Annualised at risk      : {summary['annual_at_risk']:>12,.0f}")
print(f"High-risk customers     : {summary['high_risk_count']:,} of {summary['total_customers']:,}")

In [ ]:
# Risk tier distribution chart
dist = summary['dist']
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(dist.index, dist.values)
ax.set_xlabel('% of Customers')
ax.set_title('Churn Risk Distribution')
plt.tight_layout()
plt.show()

## 9. Export High-Risk List  (optional)

Save the high-risk cohort for the retention team.

In [ ]:
high_risk = scored[scored['risk_tier'] == 'High (>60%)'].copy()

# Uncomment to save:
# high_risk.to_csv('../data/processed/high_risk_customers.csv', index=False)

print(f'{len(high_risk):,} high-risk customers')
high_risk.head()